In [1]:
import pandas as pd
import numpy as np
import openpyxl
import re
from pathlib import Path
from datetime import date, timedelta
from natsort import natsort_keygen, natsorted, ns
from xlsxwriter.utility import xl_col_to_name
from datetime import datetime

pd.options.display.max_columns = None  # type: ignore
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
# ==== USER CONFIG ====
project = "MarketRMS"

start_date = date(2025, 10, 26)  # <- report generated since that date

today = date.today()

# today = "2026-02-27"
# today = datetime.strptime(today, "%Y-%m-%d").date()

DAYS_BACK = (today - start_date).days + 1

# Paths
BASE_DIR = Path("..") / "data"
HIST_SNAPSHOT_PATH = BASE_DIR / f"{project}_work_item_snapshot.parquet"
LIVE_SNAP_DIR = BASE_DIR / "live_snapshots"

# Categories
leaf_types = ["Task", "Bug"]
removed_state = "Removed"
doneish_statuses = {"Code Review", "Ready for UAT", "Ready for Prod", "In UAT", "Closed"}
daily_charge_output_path = BASE_DIR / f"{project}_daily_charge_report.xlsx"
today_str = today.strftime("%Y-%m-%d")
output_file = BASE_DIR / f"{project}_ProjectStatus_{today_str}_ADO_API.xlsx"
output_file_2 = BASE_DIR / f"{project}_Utilisation_{today_str}.xlsx"
burndown_file = BASE_DIR / f"{project}_Burndown_{today_str}.xlsx"

# Prepare shapshots dataset for analysis

In [3]:
# Load historical snapshots with latest X–1 days
hist = pd.read_parquet(HIST_SNAPSHOT_PATH)

# Normalize snapshot date for filtering
hist["SnapshotDate"] = pd.to_datetime(hist["SnapshotDate"], utc=True, errors="coerce").dt.date

# Filter days
last_hist_date = hist["SnapshotDate"].max()
days_from_hist = max(DAYS_BACK, 0)

if days_from_hist > 0:
    hist_cutoff = last_hist_date - timedelta(days=days_from_hist - 1)
    hist_recent = hist[hist["SnapshotDate"] >= hist_cutoff].copy()
else:
    hist_recent = hist.iloc[0:0].copy()  # empty frame with same schema

print(f"Historical data selected: {hist_recent.SnapshotDate.min()} - {hist_recent.SnapshotDate.max()}")

Historical data selected: 2025-10-25 - 2026-03-29


In [ ]:
# Load today snapshot
# Find latest live snapshot file
LIVE_SNAP_DIR.mkdir(exist_ok=True)  # ensure folder exists
live_files = sorted(LIVE_SNAP_DIR.glob(f"{project}_live_snapshot_*.parquet"))

if not live_files:
    print("No live snapshot files found in", LIVE_SNAP_DIR)
    live_df = pd.DataFrame()
    live_date = None
else:
    # infer date from filename: {project}_live_snapshot_YYYY-MM-DD.parquet
    def extract_date_from_name(p: Path):
        m = re.search(r"_live_snapshot_(\d{4}-\d{2}-\d{2})\.parquet", p.name)
        return pd.to_datetime(m.group(1)).date() if m else date.min

    latest_file = max(live_files, key=extract_date_from_name)
    live_date = extract_date_from_name(latest_file)

    print(f"Latest live snapshot file: {latest_file.name} (date={live_date})")

    live_df = pd.read_parquet(latest_file)
    # Add SnapshotDate column to live data
    live_df["SnapshotDate"] = live_date

    print("Live rows loaded:", len(live_df))

In [5]:
# Combine history (last X-1 days) with latest live snapshot (today)
frames = []

if not hist_recent.empty:
    frames.append(hist_recent)

if not live_df.empty:
    if not hist_recent.empty:
        # Only keep columns that exist in hist_recent
        live_df = live_df.reindex(columns=hist_recent.columns, fill_value=pd.NA)
    frames.append(live_df)

# Drop empty frames and remove columns that are entirely NaN
cleaned_frames = []
for f in frames:
    if not f.empty:
        f = f.loc[:, f.notna().any(axis=0)]  # drop all-NaN columns
        if not f.empty:
            cleaned_frames.append(f)

if cleaned_frames:
    dataset = pd.concat(cleaned_frames, ignore_index=True)
else:
    dataset = pd.DataFrame()

print("Final dataset date range:",
      dataset["SnapshotDate"].min() if "SnapshotDate" in dataset.columns and not dataset.empty else None,
      "→",
      dataset["SnapshotDate"].max() if "SnapshotDate" in dataset.columns and not dataset.empty else None)

dataset["SnapshotDate"] = pd.to_datetime(dataset["SnapshotDate"], errors="coerce")

Final dataset date range: 2025-10-25 → 2026-03-30


In [6]:
# Minor data reformating

for c in ["OriginalEstimate", "CompletedWork", "RemainingWork"]:
    dataset[c] = pd.to_numeric(dataset[c], errors="coerce").fillna(0.0)

for c in ["ParentWorkItemId", "Priority"]:
    dataset[c] = pd.to_numeric(dataset[c], errors="coerce").astype("Int64")

dataset["IterationStart"] = pd.to_datetime(dataset["IterationStart"], errors="coerce").dt.date
dataset["IterationEnd"] = pd.to_datetime(dataset["IterationEnd"], errors="coerce").dt.date

In [7]:
# Remove "trash" tasks:
# -----------------------------
trash_tasks = (
    dataset["WorkItemType"].isin(["Task", "Bug"])
    & (dataset["State"] == removed_state)
    & (dataset["OriginalEstimate"] == 0)
    & (dataset["CompletedWork"] == 0)
)
dataset = dataset[~trash_tasks]

In [8]:
# Calculate current and previous sprints
sprints = dataset[["IterationName", "IterationStart", "IterationEnd"]].drop_duplicates().sort_values("IterationStart")

mask_current = (sprints["IterationStart"] <= today) & (sprints["IterationEnd"] >= today)
current_row = sprints.loc[mask_current].sort_values("IterationStart").tail(1)
if not current_row.empty:
    current_sprint = current_row.iloc[0]["IterationName"]
    current_start  = current_row.iloc[0]["IterationStart"]
else:
    current_sprint = None
    current_start  = None

if current_start is not None:
    prev_mask = sprints["IterationEnd"] < current_start
else:
    prev_mask = sprints["IterationEnd"] < today

prev_row = sprints.loc[prev_mask].sort_values("IterationEnd").tail(1)

previous_sprint = prev_row.iloc[0]["IterationName"] if not prev_row.empty else None

previous_sprint, current_sprint

('Sprint 35 (16.03-27.03)', None)

# Charge by day report

In [9]:
df = dataset.copy()

# Ensure SnapshotDate is a date (not string)
df["SnapshotDate"] = pd.to_datetime(df["SnapshotDate"], errors="coerce").dt.date
# For safety: keep only one snapshot per work item per date (latest revision)
df = df.sort_values(["WorkItemId", "SnapshotDate", "WorkItemRevisionSK"])
df = df.drop_duplicates(["WorkItemId", "SnapshotDate"], keep="last")

# Daily charge = difference in CompletedWork vs previous day for that work item
df["daily_charge"] = (
    df.groupby("WorkItemId")["CompletedWork"]
      .diff()
      .fillna(df["CompletedWork"])  # first day uses its own value
)

# Optional: avoid negative values (reopened / manual corrections)
# df["daily_charge"] = df["daily_charge"].clip(lower=0)

In [10]:
# --- Calculate Daily charge by Assignee ---
agg1 = (
    df.groupby(["AssignedTo", "SnapshotDate"], dropna=False)["daily_charge"]
      .sum()
      .reset_index()
)

# Replace NaN assignees with something readable
agg1["AssignedTo"] = agg1["AssignedTo"].fillna("Unassigned")

# Pivot to wide format (dates as columns), numeric fill
daily_charge_by_person = (
    agg1.pivot(index="AssignedTo", columns="SnapshotDate", values="daily_charge")
        .fillna(0)
)
daily_charge_by_person.columns.name = None  # <- remove "SnapshotDate" label

# Sort date columns
daily_charge_by_person = daily_charge_by_person.sort_index(axis=1)

# 1) Drop the earliest date column (to avoid incoming hours)
earliest_col = daily_charge_by_person.columns.min()
daily_charge_by_person = daily_charge_by_person.drop(columns=[earliest_col])

# 2) Compute total_charge from remaining date columns
daily_charge_by_person["total_charge"] = daily_charge_by_person.sum(axis=1)

# 3) Drop rows with zero total charge
daily_charge_by_person = daily_charge_by_person[
    daily_charge_by_person["total_charge"] != 0
]

# 4) Bring AssignedTo back as a column
daily_charge_by_person.reset_index(inplace=True)
daily_charge_by_person.rename(columns={"AssignedTo": "Assigned To"}, inplace=True)

# 5) Convert date columns to strings for Excel
#    (now columns are: "Assigned To", <dates>, "total_charge")
date_cols = [
    c for c in daily_charge_by_person.columns
    if c not in ("Assigned To", "total_charge")
]
rename_date_cols = {c: c.strftime("%Y-%m-%d") for c in date_cols}
daily_charge_by_person.rename(columns=rename_date_cols, inplace=True)

In [11]:
# --- Calculate Daily charge by Task ---
agg2 = (
    df.groupby(
        ["AssignedTo", "IterationName", "WorkItemType", "Title", "WorkItemURL", "SnapshotDate"],
        dropna=False
    )["daily_charge"]
    .sum()
    .reset_index()
)

agg2["AssignedTo"] = agg2["AssignedTo"].fillna("Unassigned")

# Pivot to wide format (dates as columns)
daily_charge_by_task = (
    agg2.pivot(
        index=["AssignedTo", "IterationName", "WorkItemType", "Title", "WorkItemURL"],
        columns="SnapshotDate",
        values="daily_charge"
    )
    .fillna(0)
)

# Sort date columns
daily_charge_by_task = daily_charge_by_task.sort_index(axis=1)

# Drop the earliest date column (to avoid incoming hours)
earliest_col = daily_charge_by_task.columns.min()
daily_charge_by_task = daily_charge_by_task.drop(columns=[earliest_col])

# Add total_charge from remaining date columns
daily_charge_by_task["total_charge"] = daily_charge_by_task.sum(axis=1)

# Drop rows with zero total charge
daily_charge_by_task = daily_charge_by_task[
    daily_charge_by_task["total_charge"] != 0
]

# Identify date columns (still datetime.date at this point)
date_cols = [c for c in daily_charge_by_task.columns if c != "total_charge"]

# Rename date columns to strings for Excel (YYYY-MM-DD)
rename_dates = {d: d.strftime("%Y-%m-%d") for d in date_cols}
daily_charge_by_task = daily_charge_by_task.rename(columns=rename_dates)

# Bring index back as columns
daily_charge_by_task.reset_index(inplace=True)

# Rename for final schema
daily_charge_by_task.rename(columns={"AssignedTo": "Assigned To"}, inplace=True)

# Final column ordering
base_cols = ["Assigned To", "IterationName", "WorkItemType", "Title"]
date_cols_str = [rename_dates[d] for d in date_cols]  # in sorted order
final_cols = base_cols + date_cols_str + ["total_charge", "WorkItemURL"]

# Keep only columns that actually exist (safety)
final_cols = [c for c in final_cols if c in daily_charge_by_task.columns]
daily_charge_by_task = daily_charge_by_task[final_cols]

In [ ]:
# --- Download Daily charge for Power BI ---

# 1) Identify date columns (everything except ID + total)
date_cols_person = [
    c for c in daily_charge_by_person.columns
    if c not in ("Assigned To", "total_charge")
]

# 2) Melt only date columns
daily_charge_by_person_bi = daily_charge_by_person.melt(
    id_vars=["Assigned To"],
    value_vars=date_cols_person,
    var_name="Date",
    value_name="Hours"
)

# 3) Parse dates and derive MonthYear / DayNum
daily_charge_by_person_bi["Date"] = pd.to_datetime(
    daily_charge_by_person_bi["Date"],
    errors="coerce"
)

# Drop any rows where Date couldn't be parsed (just in case)
daily_charge_by_person_bi = daily_charge_by_person_bi.dropna(subset=["Date"])

daily_charge_by_person_bi["MonthYear"] = daily_charge_by_person_bi["Date"].dt.strftime("%b-%y")
daily_charge_by_person_bi["DayNum"] = daily_charge_by_person_bi["Date"].dt.day

# If you still want Date as text for Power BI:
daily_charge_by_person_bi["Date"] = daily_charge_by_person_bi["Date"].dt.strftime("%Y-%m-%d")

daily_charge_by_person_bi.to_csv(BASE_DIR / f"{project}_PowerBI_daily_charge.csv", index=False)

# Report for Daily (Stand-Ups)

In [13]:
# --- 0. Basic prep ---

# Ensure SnapshotDate is date type
dataset_copy = dataset.copy()
dataset_copy["SnapshotDate"] = pd.to_datetime(dataset["SnapshotDate"], errors="coerce").dt.date

# Select 3 business days (including last_snap if weekday)
last_snap = dataset_copy["SnapshotDate"].max()
workdays = []
cur = last_snap
while len(workdays) < 3:
    if cur.weekday() < 5:  # Monday=0 .. Sunday=6
        workdays.append(cur)
    cur -= timedelta(days=1)
workdays = sorted(workdays)
d_earliest, d_mid, d_latest = workdays[0], workdays[1], workdays[2]

col_completed_earliest = f"Completed for {d_earliest}"
col_completed_mid      = f"Completed for {d_mid}"
col_completed_latest   = f"Completed for {d_latest}"

# Snapshot "as of today" (last_snap) ---
today_snap = dataset_copy[dataset_copy["SnapshotDate"] == last_snap].copy()

# Build quick lookup for parent relationships
id_map = today_snap.set_index("WorkItemId")[["WorkItemType", "Title", "ParentWorkItemId"]]

# Identify leaf-level work items ---

# True leaves = items that are not a parent of any other item + workitems with time estimates or charges
children_parent_ids = set(today_snap["ParentWorkItemId"].dropna().astype(int))
leaves = today_snap[
    (today_snap["WorkItemType"].isin({"Task", "Bug"})) |
    (
            (today_snap["CompletedWork"] != 0) |
            (today_snap["OriginalEstimate"] != 0) |
            (today_snap["RemainingWork"] != 0)
        )
].copy()

In [14]:
# --- Parent Chain function ---

def parent_chain(row):
    wid = row["WorkItemId"]
    wtype = row["WorkItemType"]
    title = row["Title"]

    story_title = None
    epic_title = None

    # If the leaf itself is a User Story, use it as story
    if wtype in {"User Story"}:
        story_title = title

    # If the leaf itself is an Epic (just in case), treat as Epic
    if wtype == "Epic":
        epic_title = title

    visited = set()
    cur_id = wid

    # Walk up parent chain
    for _ in range(10):  # 3x takst + 3x Us + 3x Fetures + 1x Epic
        try:
            parent_id = id_map.at[cur_id, "ParentWorkItemId"]
        except KeyError:
            break

        if pd.isna(parent_id):
            break

        parent_id = int(parent_id)
        if parent_id in visited:
            break
        visited.add(parent_id)

        try:
            parent_type = id_map.at[parent_id, "WorkItemType"]
            parent_title = id_map.at[parent_id, "Title"]
        except KeyError:
            break

        if parent_type in {"User Story"} and story_title is None:
            story_title = parent_title

        if parent_type == "Epic":
            # always update so we end up with the top-most Epic
            epic_title = parent_title

        cur_id = parent_id

    return pd.Series({"User Story": story_title, "Epic": epic_title})

In [15]:
# --- 1. Parent Chain ---

hier = leaves.apply(parent_chain, axis=1)
leaves = pd.concat([leaves, hier], axis=1)

# If leaf itself is Story and still missing, fill from Title
mask_story_self = leaves["WorkItemType"].isin({"User Story"}) & leaves["User Story"].isna()
leaves.loc[mask_story_self, "User Story"] = leaves.loc[mask_story_self, "Title"]

In [16]:
# --- 2. Completed time for last 3 business days (from df with daily_charge) ---

# Melt daily_charge_by_task into long form

date_cols = [
    c for c in daily_charge_by_person.columns
    if c not in ("Assigned To", "total_charge") and c[:4].isdigit()
]

daily_long = daily_charge_by_task.melt(
    id_vars=["WorkItemURL"],          # we only need this to merge later
    value_vars=date_cols,
    var_name="Date",
    value_name="daily_charge"
)
daily_long["Date"] = pd.to_datetime(daily_long["Date"], errors="coerce")

# Filter period for last 3 working days

workdays = []
cur = today

if today.weekday() < 3: # Mon–Wed
    while len(workdays) < 5:
        workdays.append(cur)
        cur -= timedelta(days=1)
else:
    while len(workdays) < 3:
        workdays.append(cur)
        cur -= timedelta(days=1)

workdays = list(set(workdays))
daily_long = daily_long[daily_long["Date"].isin(pd.to_datetime(workdays))]

# Add column with working days date (for weekends -> monday)

daily_long["WorkDate"] = daily_long["Date"] + pd.to_timedelta(
    daily_long["Date"].dt.weekday.map({5: 2, 6: 1}).fillna(0), unit="D"
)

# Prepare charged hours to merge
stand_up_charge = daily_long.groupby(["WorkItemURL", "WorkDate"], as_index=False).agg(
          {
            "daily_charge": "sum",
          }
        ).reset_index(drop=True)

stand_up_charge = stand_up_charge.pivot(index="WorkItemURL", columns="WorkDate", values="daily_charge").reset_index(drop = False)

# Prepare column names and rename
dates_to_present = sorted(list(daily_long["WorkDate"].unique()))
d_earliest, d_mid, d_latest = dates_to_present

col_completed_earliest = f"Completed for {d_earliest.date()}"
col_completed_mid = f"Completed for {d_mid.date()}"
col_completed_latest = f"Completed for {d_latest.date()}"

rename_map = {
    d_earliest: col_completed_earliest,
    d_mid:      col_completed_mid,
    d_latest:   col_completed_latest,
}

stand_up_charge = stand_up_charge.rename(columns=rename_map)

# Merge charges to dataframe
daily_meeting = leaves.merge(
    stand_up_charge,
    on = "WorkItemURL",
    how = "left"
)

for c in [col_completed_earliest, col_completed_mid, col_completed_latest]:
    if c in daily_meeting.columns:
        daily_meeting[c] = daily_meeting[c].fillna(0.0)


In [17]:
# --- 3. Final filtering ---

daily_meeting = daily_meeting[[
    "IterationName",
    "Epic",
    "User Story",
    "Title",
    "Priority",
    "AssignedTo",
    "State",
    "OriginalEstimate",
    "CompletedWork",
    col_completed_earliest,
    col_completed_mid,
    col_completed_latest,
    "RemainingWork",
    "WorkItemURL",
    "IterationStart",
    "IterationEnd"
]].copy()

# masks
state_closed = daily_meeting["State"].isin(["Closed", "Removed"])

in_current_sprint = (
    (daily_meeting["IterationStart"] <= today) &
    (daily_meeting["IterationEnd"] >= today)
)
not_current_sprint = ~in_current_sprint

# ensure NaN treated as 0 for the numeric checks
num_cols = [col_completed_earliest, col_completed_mid, col_completed_latest, "RemainingWork"]
daily_meeting[num_cols] = daily_meeting[num_cols].fillna(0)

all_zero_activity = (
    (daily_meeting[col_completed_earliest] == 0) &
    (daily_meeting[col_completed_mid] == 0) &
    (daily_meeting[col_completed_latest] == 0) &
    (daily_meeting["RemainingWork"] == 0)
)

# rows to drop
to_drop = state_closed & not_current_sprint & all_zero_activity

# apply filter
daily_meeting_filtered = daily_meeting[~to_drop].copy()

# sorting
natkey = natsort_keygen(alg=ns.IGNORECASE)
daily_meeting_filtered = (
    daily_meeting_filtered
    .sort_values(
        by=["IterationStart", "IterationName", "Epic", "User Story", "Priority"],
        ascending=[True, True, True, True, True],
        key=lambda col: col.astype(str).map(natkey)
    )
    .reset_index(drop=True)
).drop(columns = ["IterationStart", "IterationEnd"])

# Sprint Burndown

In [18]:
# --- Burndown calculation function ---
def sprint_burndown(df_leaf, sprint_name):
    """
    Return DataFrame with columns: Date, Remaining Work
    for snapshots within the selected sprint window:
      IterationStart <= Snapshot Date <= (IterationEnd - 1 day).
    Assumes df_leaf has columns:
      - Sprint (IterationName), Snapshot Date, State,
        Remaining Work, IterationStart, IterationEnd.
    """
    if not sprint_name:
        return pd.DataFrame(columns=["Date", "RemainingWork"])
    
    sub = df_leaf.loc[df_leaf["IterationName"] == sprint_name].copy()
    
    if sub.empty:
        return pd.DataFrame(columns=["Date", "RemainingWork"])

    # Resolve sprint dates from the rows (handle tz + ensure date-only)
    start = pd.to_datetime(sub["IterationStart"], errors="coerce")
    end   = pd.to_datetime(sub["IterationEnd"], errors="coerce")

    if start.notna().any():
        sprint_start = start
        sprint_start = sprint_start
    else:
        return pd.DataFrame(columns=["Date", "RemainingWork"])

    if end.notna().any():
        sprint_end_exclusive = end
        # include up to End - 1 day
        sprint_last_inclusive = sprint_end_exclusive - pd.Timedelta(days=1)
    else:
        return pd.DataFrame(columns=["Date", "RemainingWork"])

    # Keep snapshots only within sprint window
    snap = pd.to_datetime(sub["SnapshotDate"], errors="coerce")
    if getattr(snap.dt, "tz", None) is not None:
        snap = snap.dt.tz_convert(None)
    sub = sub.loc[(snap >= sprint_start) & (snap <= sprint_last_inclusive)].copy()
    if sub.empty:
        return pd.DataFrame(columns=["Date", "RemainingWork"])

    # Exclude removed items from burndown
    sub = sub[sub["State"] != removed_state]

    # Clear remaining work for items in done-ish states
    sub["RemainingBD"] = sub["RemainingWork"]
    sub.loc[sub["State"].isin(doneish_statuses), "RemainingBD"] = 0.0

    bd = (
        sub.groupby(pd.to_datetime(sub["SnapshotDate"]).dt.normalize())["RemainingBD"]
           .sum()
           .sort_index()
           .reset_index(name="Date")
           .rename(columns={"RemainingBD": "RemainingWork"})
    )
    return bd

In [ ]:
# --- 1. Calculate BD for current and previous Iterations ---
bd_current = sprint_burndown(dataset, current_sprint)
bd_prev = sprint_burndown(dataset, previous_sprint)

In [20]:
# # --- 2. Calculate BD by assignee for current Iteration ---
dataset["SnapshotDate"] = pd.to_datetime(dataset["SnapshotDate"], utc=True).dt.tz_localize(None)
leaf_curr = dataset[
    (dataset["IterationName"] == current_sprint)
    & (dataset["SnapshotDate"].isin(bd_current.SnapshotDate.unique()))
].copy()

# Exclude 'Removed' from burndown
leaf_curr = leaf_curr[leaf_curr["State"] != removed_state]

# Apply your burndown rules
leaf_curr["RemainingBD"] = leaf_curr["RemainingWork"]
leaf_curr.loc[leaf_curr["State"].isin(doneish_statuses), "RemainingBD"] = 0.0

# Group by Snapshot Date + Assignee
bd_assignee = (
    leaf_curr.groupby(["SnapshotDate", "AssignedTo"])["RemainingBD"]
             .sum()
             .reset_index()
)

# Pivot: one column per assignee
bd_assignee_pivot = (
    bd_assignee
    .pivot(index="SnapshotDate", columns="AssignedTo", values="RemainingBD")
    .fillna(0.0)
    .sort_index()
)

# Flatten index/columns for Excel
bd_assignee_pivot = bd_assignee_pivot.reset_index().rename(columns={"SnapshotDate": "Date"})
assignee_cols = [c for c in bd_assignee_pivot.columns if c != "Date"]

In [21]:
from xlsxwriter.utility import xl_col_to_name
import pandas as pd

# --- Prep summary (stand-up) table ---
summary = daily_meeting_filtered.copy()

# Strip timezone for Excel if present
for col in ["IterationStart", "IterationEnd"]:
    if col in summary.columns and pd.api.types.is_datetime64_any_dtype(summary[col]):
        try:
            summary[col] = summary[col].dt.tz_convert(None)
        except AttributeError:
            pass

# Columns to highlight in blue
completed_cols = [col_completed_earliest, col_completed_mid, col_completed_latest]

with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
    # --- Write sheets in desired order ---

    # 1) Status
    summary.to_excel(writer, index=False, sheet_name="Status")

    # 2) CurrentByAssignee (if exists)
    if not bd_assignee_pivot.empty:
        bd_assignee_pivot.to_excel(writer, index=False, sheet_name="CurrentByAssignee")

    # 3) Charge_Person
    daily_charge_by_person.to_excel(writer, sheet_name="Charge_Person", index=False)

    # 4) Charge_Task
    daily_charge_by_task.to_excel(writer, sheet_name="Charge_Task", index=False)

    # 5) CurrentSprint (if exists)
    if not bd_current.empty:
        bd_current.to_excel(writer, index=False, sheet_name="CurrentSprint")

    # 6) PreviousSprint (if exists)
    if not bd_prev.empty:
        bd_prev.to_excel(writer, index=False, sheet_name="PreviousSprint")

    # --- Workbook & base sheets ---
    wb = writer.book
    ws_status = writer.sheets["Status"]
    ws_by_person = writer.sheets["Charge_Person"]
    ws_by_task = writer.sheets["Charge_Task"]

    # ---------- Common formats ----------
    number_fmt = wb.add_format({'num_format': '#,##0.0;-#,##0.0;"-"'})
    percent_fmt = wb.add_format({"num_format": "0.0%"})
    header_fmt = wb.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'align': 'center',
        'border': 1
    })
    header_fmt_blue = wb.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'align': 'center',
        'border': 1,
        'font_color': '#0070C0'
    })
    blue_number_fmt = wb.add_format({
        'num_format': '#,##0.0;-#,##0.0;"-"',
        'font_color': '#0070C0'
    })
    header_horizontal = wb.add_format({
        "bold": True, "align": "center", "valign": "top",
        "text_wrap": False, "border": 1
    })
    header_vertical = wb.add_format({
        "bold": True,
        "align": "center",
        "valign": "bottom",
        "rotation": 90,
        "text_wrap": False,
        "border": 1
    })
    weekend_fill = wb.add_format({
        "bg_color": "#EAB8D8",
        "align": "center",
        "valign": "bottom",
        "rotation": 90,
        "text_wrap": False,
        "border": 1
    })
    weekend_number_fmt = wb.add_format({
        "num_format": '#,##0.0;-#,##0.0;"-"',
        "bg_color": "#EAB8D8",
        "align": "center",
        "valign": "vcenter",
    })

    # ========== STATUS sheet formatting ==========
    for col_idx, col_name in enumerate(summary.columns):
        width = 15
        fmt = None

        if col_name == "IterationName":
            width = 22
        elif col_name == "Epic":
            width = 22
        elif col_name == "User Story":
            width = 40
        elif col_name == "Title":
            width = 60
        elif col_name == "Priority":
            width = 7
        elif col_name in ("AssignedTo", "State"):
            width = 18
        elif col_name in ("OriginalEstimate", "CompletedWork", "RemainingWork"):
            width = 15
            fmt = number_fmt
        elif col_name in completed_cols:
            width = 15
            fmt = blue_number_fmt
        elif col_name == "WorkItemURL":
            width = 60
        elif col_name in ("IterationStart", "IterationEnd"):
            width = 18

        ws_status.set_column(col_idx, col_idx, width, fmt)

    for col_num, header in enumerate(summary.columns):
        fmt = header_fmt_blue if header in completed_cols else header_fmt
        ws_status.write(0, col_num, header.replace(" ", "\n"), fmt)
    ws_status.set_row(0, 47)
    ws_status.freeze_panes(1, 0)

    if "State" in summary.columns:
        col_state = summary.columns.get_loc("State")
        col_letter = xl_col_to_name(col_state)
        blocked_fmt = wb.add_format({'bg_color': '#FFCDCE'})

        first_row = 1
        last_row = len(summary)
        first_col = 0
        last_col = len(summary.columns) - 1

        ws_status.conditional_format(
            first_row, first_col, last_row, last_col,
            {
                'type': 'formula',
                'criteria': f'=${col_letter}{first_row+1}="Blocked"',
                'format': blocked_fmt
            }
        )

    # ========== BY PERSON sheet formatting ==========
    ws_by_person.set_column(0, 0, 22)  # Assigned To
    ws_by_person.set_column(1, len(daily_charge_by_person.columns) - 2, 4, number_fmt)
    ws_by_person.set_column(
        len(daily_charge_by_person.columns) - 1,
        len(daily_charge_by_person.columns) - 1,
        12,
        number_fmt
    )

    date_cols_person = [
        c for c in daily_charge_by_person.columns
        if c not in ("Assigned To", "total_charge") and isinstance(c, str) and c[:4].isdigit()
    ]

    for col_idx, col_name in enumerate(daily_charge_by_person.columns):
        is_date_col = col_name in date_cols_person
        fmt_header = header_vertical if is_date_col else header_horizontal
        fmt_data = number_fmt if is_date_col else None

        if is_date_col:
            try:
                dt = pd.to_datetime(col_name).date()
                if dt.weekday() >= 5:
                    fmt_header = weekend_fill
                    fmt_data = weekend_number_fmt
            except Exception:
                pass

        ws_by_person.write(0, col_idx, col_name, fmt_header)

        if is_date_col:
            for row_idx, cell_value in enumerate(daily_charge_by_person[col_name], start=1):
                ws_by_person.write(row_idx, col_idx, cell_value, fmt_data)

    ws_by_person.set_row(0, 61)
    ws_by_person.freeze_panes(1, 1)

    # ========== BY TASK sheet formatting ==========
    ws_by_task.set_column(0, 1, 22)
    ws_by_task.set_column(2, 2, 15)
    ws_by_task.set_column(3, 3, 40)
    ws_by_task.set_column(4, len(daily_charge_by_task.columns) - 3, 4, number_fmt)
    ws_by_task.set_column(
        len(daily_charge_by_task.columns) - 2,
        len(daily_charge_by_task.columns) - 1,
        12,
        number_fmt
    )

    date_cols_task = [
        c for c in daily_charge_by_task.columns
        if c not in ("Assigned To", "IterationName", "WorkItemType", "Title", "total_charge", "WorkItemURL")
        and isinstance(c, str) and c[:4].isdigit()
    ]

    for col_idx, col_name in enumerate(daily_charge_by_task.columns):
        is_date_col = col_name in date_cols_task
        fmt_header = header_vertical if is_date_col else header_horizontal
        fmt_data = number_fmt if is_date_col else None

        if is_date_col:
            try:
                dt = pd.to_datetime(col_name).date()
                if dt.weekday() >= 5:
                    fmt_header = weekend_fill
                    fmt_data = weekend_number_fmt
            except Exception:
                pass

        ws_by_task.write(0, col_idx, col_name, fmt_header)

        if is_date_col:
            for row_idx, cell_value in enumerate(daily_charge_by_task[col_name], start=1):
                ws_by_task.write(row_idx, col_idx, cell_value, fmt_data)

    ws_by_task.set_row(0, 61)
    ws_by_task.freeze_panes(1, 4)

    # ========== BURNDOWN SHEETS & CHARTS ==========

    # 1) Current sprint burndown
    if not bd_current.empty:
        ws_curr = writer.sheets["CurrentSprint"]

        for col_idx, col_name in enumerate(bd_current.columns):
            ws_curr.write(0, col_idx, col_name, header_fmt)
        ws_curr.freeze_panes(1, 0)

        chart_curr = wb.add_chart({"type": "line"})
        max_row_curr = len(bd_current)
        chart_curr.add_series({
            "name":       f"Burndown - {current_sprint}",
            "categories": ["CurrentSprint", 1, 0, max_row_curr, 0],
            "values":     ["CurrentSprint", 1, 1, max_row_curr, 1],
        })
        chart_curr.set_title({"name": f"Burndown - {current_sprint}"})
        chart_curr.set_x_axis({"name": "Date"})
        chart_curr.set_y_axis({"name": "Remaining Work (hours)"})
        ws_curr.insert_chart("D2", chart_curr)

    # 2) Previous sprint burndown
    if not bd_prev.empty:
        ws_prev = writer.sheets["PreviousSprint"]

        for col_idx, col_name in enumerate(bd_prev.columns):
            ws_prev.write(0, col_idx, col_name, header_fmt)
        ws_prev.freeze_panes(1, 0)

        chart_prev = wb.add_chart({"type": "line"})
        max_row_prev = len(bd_prev)
        chart_prev.add_series({
            "name":       f"Burndown - {previous_sprint}",
            "categories": ["PreviousSprint", 1, 0, max_row_prev, 0],
            "values":     ["PreviousSprint", 1, 1, max_row_prev, 1],
        })
        chart_prev.set_title({"name": f"Burndown - {previous_sprint}"})
        chart_prev.set_x_axis({"name": "Date"})
        chart_prev.set_y_axis({"name": "Remaining Work (hours)"})
        ws_prev.insert_chart("D2", chart_prev)

    # 3) Current sprint by assignee: line + colored histogram
    if not bd_assignee_pivot.empty:
        ws_ass = writer.sheets["CurrentByAssignee"]

        for col_idx, col_name in enumerate(bd_assignee_pivot.columns):
            ws_ass.write(0, col_idx, col_name, header_fmt)
        ws_ass.freeze_panes(1, 1)

        max_row_ass = len(bd_assignee_pivot)

        # Line chart per assignee
        chart_ass = wb.add_chart({"type": "line"})
        for col_idx, col_name in enumerate(bd_assignee_pivot.columns[1:], start=1):
            chart_ass.add_series({
                "name":       ["CurrentByAssignee", 0, col_idx],
                "categories": ["CurrentByAssignee", 1, 0, max_row_ass, 0],  # Date
                "values":     ["CurrentByAssignee", 1, col_idx, max_row_ass, col_idx],
            })

        chart_ass.set_title({"name": f"Burndown by Assignee - {current_sprint}"})
        chart_ass.set_x_axis({"name": "Date"})
        chart_ass.set_y_axis({"name": "Remaining Work (hours)"})
        ws_ass.insert_chart("H2", chart_ass)

        # Completion % row
        progress_row = max_row_ass + 2
        ws_ass.write(progress_row, 0, "Completion %", header_fmt)

        for col_idx, col_name in enumerate(bd_assignee_pivot.columns[1:], start=1):
            start_val = bd_assignee_pivot.iloc[0, col_idx]
            current_val = bd_assignee_pivot.iloc[-1, col_idx]

            if pd.notna(start_val) and start_val != 0:
                pct = (start_val - current_val) / start_val
                pct = max(0.0, min(1.0, pct))
            else:
                pct = 0.0

            ws_ass.write_number(progress_row, col_idx, pct, percent_fmt)

        # Colored histogram
        chart_hist = wb.add_chart({"type": "column"})
        num_assignees = len(bd_assignee_pivot.columns) - 1

        colors = [
            "#4472C4",  # blue
            "#ED7D31",  # orange
            "#A5A5A5",  # gray
            "#FFC000",  # yellow
            "#5B9BD5",  # light blue
            "#70AD47",  # green
        ]
        points = [
            {"fill": {"color": colors[i % len(colors)]}}
            for i in range(num_assignees)
        ]

        chart_hist.add_series({
            "name":       "Completion %",
            "categories": ["CurrentByAssignee", 0, 1, 0, num_assignees],
            "values":     ["CurrentByAssignee", progress_row, 1, progress_row, num_assignees],
            "points":     points,
        })

        chart_hist.set_title({"name": f"Assignee Completion - {current_sprint}"})
        chart_hist.set_y_axis({
            "name": "Completion",
            "min": 0,
            "max": 1,
            "major_unit": 0.25,
            "num_format": "0%",
        })

        ws_ass.insert_chart("H20", chart_hist)

print(f"✅ Done! File saved: {output_file}")


✅ Done! File saved: MRMS_ProjectStatus_2026-03-30_ADO_API.xlsx


## 2. Utilisation calculation

In [22]:
# Prepare data for calculation

sprints = sprints[
    (~sprints['IterationStart'].isna()) &
    (~sprints['IterationEnd'].isna())
]
sprints['IterationStart'] = pd.to_datetime(sprints["IterationStart"])
sprints['IterationEnd'] = pd.to_datetime(sprints["IterationEnd"])

# Filter source whole dataset for utilisation calculation
utilisation = dataset[["SnapshotDate", "IterationName", "IterationStart", "IterationEnd", "WorkItemType", "Title", "State", "AssignedTo", "CompletedWork", "OriginalEstimate", "RemainingWork", "WorkItemURL"]].copy()
utilisation['IterationStart'] = pd.to_datetime(utilisation["IterationStart"])
utilisation['IterationEnd'] = pd.to_datetime(utilisation["IterationEnd"])
utilisation = utilisation[utilisation["IterationStart"] >= pd.to_datetime(start_date)]

# Select Snapshots at the first day of each Sprint (it should be day's night, so Snapshot is after planning)
utilisation = utilisation[utilisation["SnapshotDate"] == utilisation["IterationStart"]]

# Calculate CompletedWork at the start of sprint
utilisation["PrevDate"] = utilisation["SnapshotDate"] - pd.Timedelta(days=1)
utilisation = utilisation.merge(
    dataset[["SnapshotDate", "WorkItemURL", "CompletedWork"]],
    left_on=["WorkItemURL", "PrevDate"],
    right_on=["WorkItemURL", "SnapshotDate"],
    how="left",
    suffixes=('', '_start')
).drop(columns = ["SnapshotDate_start", "PrevDate"])
utilisation["Completed_at_sprint_start"] = utilisation["CompletedWork_start"].fillna(0)
utilisation = utilisation.drop(columns = ["CompletedWork", "CompletedWork_start"])

# Calculate Planned_for_sprint
utilisation["Planned_for_sprint"] = (utilisation["OriginalEstimate"] - utilisation["Completed_at_sprint_start"]).clip(lower=0)

### 2-1 Charged time by sprints

In [23]:
# Charged time by sprints

# 1 Melt table with charges
id_cols = ["Assigned To", "IterationName", "WorkItemType", "Title", "WorkItemURL"]
date_cols = [c for c in daily_charge_by_task.columns if c not in id_cols + ["total_charge"]]
charge_by_sprint = daily_charge_by_task.melt(
    id_vars=id_cols,
    value_vars=date_cols,
    var_name="SnapshotDate",
    value_name="charge"
)

charge_by_sprint["SnapshotDate"] = pd.to_datetime(charge_by_sprint["SnapshotDate"], errors="coerce")
charge_by_sprint = charge_by_sprint[charge_by_sprint["charge"].fillna(0) != 0]


# Add 'IterationStart', 'IterationEnd' to Iteration name
charge_by_sprint = charge_by_sprint.merge(
    sprints[['IterationName', 'IterationStart', 'IterationEnd']],
    how='left',
    left_on=None,
    right_on=None,
    suffixes=('', '_s')
)

# 2 Match Iteration for each charged date - WorkingIteration
intervals = pd.IntervalIndex.from_arrays(sprints['IterationStart'], sprints['IterationEnd'], closed='both')
idx = intervals.get_indexer(charge_by_sprint['SnapshotDate'])
charge_by_sprint['WorkingIteration'] = np.where(idx >= 0, sprints['IterationName'].to_numpy()[idx], pd.NA)

# 3 Group daily disagregation to Iteration disagregation
charge_by_sprint = (
    charge_by_sprint.groupby(['WorkingIteration', 'Assigned To', 'WorkItemType', 'Title', 'WorkItemURL'], as_index=False)['charge']
        .sum()
)
charge_by_sprint = charge_by_sprint.dropna(subset=['WorkingIteration'])

### 2-2 For each spring - split planned time and unplanned time

In [24]:
# Merge planned time charges to each sprtint tasks 

utilisation = utilisation.merge(
    charge_by_sprint[["WorkItemURL", "WorkingIteration", "charge"]],
    left_on = ["WorkItemURL", "IterationName"],
    right_on = ["WorkItemURL", "WorkingIteration"],
    how = "left"
)
utilisation["charge"] = utilisation["charge"].fillna(0)

# Mark them as planned
utilisation["Planned"] = "yes"

# Formatting
utilisation = utilisation[["IterationName", "AssignedTo", "WorkItemType", "Title", "Planned", "Planned_for_sprint", "charge", "WorkItemURL"]]


In [25]:
# Filter charges from 'charge_by_sprint', which were not planned for each sprint (utilisation["WorkItemURL"] does not match with charge_by_sprint["WorkingIteration"]) 
not_planned_charge = charge_by_sprint.merge(
    utilisation[["WorkItemURL", "IterationName"]],
    left_on = ["WorkItemURL", "WorkingIteration"],
    right_on = ["WorkItemURL", "IterationName"],
    how = "left"
)
not_planned_charge = not_planned_charge[not_planned_charge["IterationName"].isna()]

# Mark these as not planned
not_planned_charge["Planned"] = "no"

# Formatting
not_planned_charge["Planned_for_sprint"] = 0
not_planned_charge = not_planned_charge[["WorkingIteration", "Assigned To", "WorkItemType", "Title", "Planned_for_sprint", "charge", "WorkItemURL", "Planned"]].rename(columns = {"Assigned To": "AssignedTo", "WorkingIteration": "IterationName"})

# Combine planned tasks with not planned tasks - to single dataframe
utilisation = pd.concat([utilisation, not_planned_charge])

# Formatting
utilisation["Completed time"] = utilisation["charge"]


### 2-3 Calculate metrics

In [26]:
# Remove QA from this analysis
utilisation = utilisation[utilisation["AssignedTo"] != "Horin Serhii"]

# Select completed as planned to separate column
utilisation["Completed for planned tasks"] = np.where(
    utilisation["Planned"] == "yes",
    utilisation[["Completed time", "Planned_for_sprint"]].min(axis=1),
    0
)

# Account for underestimation
utilisation["Overcharged_planned_tasks"] = np.where(
    utilisation["Planned"] == "yes",
    (utilisation["Completed time"] - utilisation["Planned_for_sprint"]).clip(lower=0),
    0
)

# Calculate what time team failed to complete within planned sprint
utilisation["NOT DONE but planed"] = (utilisation["Planned_for_sprint"] - utilisation["Completed time"]).clip(lower=0)


# Select other completed time, which was not planned to be part of the sprint
utilisation["Completed non-sprint tasks"] = np.where(
    utilisation["Planned"] == "no",
    utilisation["Completed time"],
    0
)


In [27]:
utilisation_summary = (
    utilisation.groupby(['AssignedTo', 'IterationName'], as_index=False)
    [['Planned_for_sprint', 'Completed for planned tasks', 'NOT DONE but planed', 'Completed time', 'Completed non-sprint tasks', 'Overcharged_planned_tasks']].sum()
)

utilisation_time_map = {
    'Vasilov Denys':  60,        # 80 hours * 70%
    'Vasilov Denys S.': 60,      # 80 hours * 70%
    'Sukonnik Illia V.': 50,     # 80 hours * 60%
    'Kolodieiev Ivan': 50,       # 80 hours * 60%
    'Andrianov Vladyslav': 40,   # 80 hours * 50%
}

utilisation_summary['Utilisation time'] = (
    utilisation_summary['AssignedTo']
        .map(utilisation_time_map)
        .fillna(0)          # or raise / keep NaN
        .astype(int)
)

utilisation_summary = utilisation_summary.sort_values(["AssignedTo", "IterationName"], ascending=False)

utilisation_summary["Underutilised"] = utilisation_summary['Utilisation time'] - utilisation_summary["Completed time"]
utilisation_summary["Overplanned / Underplanned"] = utilisation_summary['Planned_for_sprint'] - utilisation_summary["Utilisation time"]
utilisation_summary["Work hours debt"] = np.maximum(np.minimum(utilisation_summary["Planned_for_sprint"], utilisation_summary["Utilisation time"]), utilisation_summary["Utilisation time"]) - utilisation_summary["Completed time"]
utilisation_summary["Comment on Plan Focus"] = ""
utilisation_summary["Comment on Utilisation"] = ""

utilisation_summary = utilisation_summary[["AssignedTo", "IterationName", "Utilisation time", "Planned_for_sprint", "Overplanned / Underplanned", "Completed for planned tasks", 'Overcharged_planned_tasks', "Completed non-sprint tasks", "Work hours debt", "Comment on Plan Focus"]]
# "Completed time", "Underutilised", "Comment on Utilisation",
# "NOT DONE but planed"

In [28]:
with pd.ExcelWriter(output_file_2, engine="xlsxwriter") as writer:
    # --- write sheets ---
    utilisation_summary.to_excel(writer, sheet_name="Summary", index=False)
    utilisation.to_excel(writer, sheet_name="Details", index=False)

    workbook  = writer.book

    header_fmt = workbook.add_format({
        "bold": True,
        "text_wrap": True,
        "valign": "top",
        "align": "center",
        "border": 1,
    })

    def format_sheet(df: pd.DataFrame, sheet_name: str, header_format, extra_pad: int = 2,
                     min_width: int = 10, max_width: int = 60):
        ws = writer.sheets[sheet_name]

        # Apply header format
        for col_idx, col_name in enumerate(df.columns):
            ws.write(0, col_idx, col_name, header_format)

        # Set column widths (based on max of header length and sampled cell lengths)
        for col_idx, col_name in enumerate(df.columns):
            # header length
            header_len = len(str(col_name))

            # content length (sample a bit for speed; adjust if you want full scan)
            if len(df) > 0:
                sample = df.iloc[:500, col_idx].astype(str)  # sample first 500 rows
                content_len = sample.map(len).max()
            else:
                content_len = 0

            width = max(header_len, content_len) + extra_pad
            width = max(min_width, min(width, max_width))

            ws.set_column(col_idx, col_idx, width)

        # Freeze header row + enable autofilter
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, 0, len(df.columns) - 1)

    format_sheet(utilisation_summary, "Summary", header_fmt)
    format_sheet(utilisation, "Details", header_fmt)

output_file_2

'MRMS_Utilisation_2026-03-30.xlsx'